# Colab GPU throughput probe

Answers one question: is whatever GPU Colab hands out today fast enough to move
Phase E (SimSiam pretraining, ~200 epochs, ~18 GPU-h estimated) off the local box?

This does **not** train on real data. It reuses `benchmarks/bench.py`'s
`measure()` unchanged — synthetic batches, no dataset download needed — at the
same settings (`176px`, `bf16`, `channels_last`, batch 256) the numbers in
`plans/2026-07-30-training-roadmap.md` were measured with, so the printed img/s
is directly comparable to the local baseline already recorded there
(1688 img/s on an RTX 5050 Laptop, 8 GB, under a ~50 W power cap).

Run top to bottom. Runtime > Change runtime type > pick a GPU first.

In [ ]:
!git clone --depth 1 https://github.com/lorenzoliuzzo/supervised-learning-on-food-images
%cd supervised-learning-on-food-images
# Colab ships torch/torchvision already; this only resolves the rest
# (pandas, pillow, torchsummary, ...) from pyproject.toml.
!pip install -q -e .

In [ ]:
import sys

sys.path.append('src')
sys.path.append('benchmarks')

import torch
from bench import TRAIN_IMAGES, measure, report

from model import FoodCNN

assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
# Same settings plans/2026-07-30-training-roadmap.md's 1688 img/s baseline used.
SIZE = 176
BATCH = 256
LOCAL_IMG_S = 1688  # RTX 5050 Laptop, 8 GB, ~50 W power cap -- see the roadmap.

result = measure(FoodCNN(num_classes=251), SIZE, BATCH)
report('FoodCNN baseline (this GPU)', result, epochs=90)

local_pretrain_hours = 200 * TRAIN_IMAGES / LOCAL_IMG_S / 3600
print(f"\n{result.images_per_second / LOCAL_IMG_S:.2f}x the local box's throughput "
      "at the same settings")
print(f"Phase E pretrain (200 ep): {result.hours(200):.1f} h here vs "
      f"{local_pretrain_hours:.1f} h locally")

## Reading the result

- **> ~1x**: worth pretraining here instead of locally — but see the session-limit
  note below before committing an 8h+ run to a free Colab runtime.
- **< ~1x**: stay local. Free-tier Colab GPUs (T4) are not guaranteed to beat a
  dedicated box once its power profile is set to `performance`
  (see the roadmap's throughput section) — this cell is what tells you which
  case you're in today, since the assigned GPU varies by session.
- Free-tier Colab sessions disconnect (~12h wall clock, sooner if idle), so a
  200-epoch run needs the resumable checkpointing `src/simsiam.py` already has
  (`--resume`, saved every epoch) — see Part 2 below before starting a real run.

---
## Part 2 (optional, not run by default): an actual pretraining run

Only relevant once the cell above says Colab is worth it. Needs `food251/`
(~5.5 GB, not in this repo) uploaded to Google Drive first — a separate,
one-time cost this notebook doesn't do for you. Checkpoints go to Drive too,
so a disconnect loses at most one epoch, not the whole run.

Skip this section entirely if you haven't uploaded the dataset yet.

In [ ]:
RUN_REAL_TRAINING = False  # flip to True once food251/ is on Drive

if RUN_REAL_TRAINING:
    from google.colab import drive
    drive.mount('/content/drive')

    DATA_DIR = '/content/drive/MyDrive/food251'       # adjust to where you uploaded it
    CHECKPOINT_DIR = '/content/drive/MyDrive/foodx-checkpoints'
    import pathlib
    pathlib.Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
    !ln -sfn {CHECKPOINT_DIR} checkpoints

    !cd src && python simsiam.py {DATA_DIR} --run-label simsiam-colab \
        --resume ../checkpoints/simsiam-colab.pth.tar